<a href="https://colab.research.google.com/github/Selma266/Malicious-URL-Detection/blob/main/Malicious_URL_Detection_2026_Final_Version_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading, Preprocessing & Dataset Generation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Chargement Phishing URLs dataset + prétraitement

In [ ]:
import pandas as pd #la bibliothèque Pandas, utilisée pour la manipulation et l’analyse de données tabulaires.
df = pd.read_csv("/content/drive/MyDrive/Phishing URLs_cleaned.csv")  # read dataset url,label  #Chargement du dataset
df.head()

,url,label
0,https://docs.google.com/presentation/d/e/2PACX...,malicious
1,https://btttelecommunniccatiion.weeblysite.com/,malicious
2,https://kq0hgp.webwave.dev/,malicious
3,https://brittishtele1bt-69836.getresponsesite....,malicious
4,https://bt-internet-105056.weeblysite.com/,malicious


Suppression des lignes redondantes

In [ ]:
# BEFORE removing duplicates
print("Before removing duplicates:")
print("Shape:", df.shape)

# Step 2: Remove duplicate rows
df = df.drop_duplicates()   #Suppression des doublons

# AFTER removing duplicates
print("\nAfter removing duplicates:")
print("Shape:", df.shape)

Before removing duplicates:
Shape: (504983, 2)

After removing duplicates:
Shape: (504933, 2)


Supprimer les NaN+ Conversion label en valeur Numeriques

In [ ]:
# Supprimer les lignes où url ou label est NaN
df = df.dropna(subset=["url", "label"]).reset_index(drop=True)  #reset_index(drop=True) Réinitialise l’index du DataFrame
# strip() supprime les espaces au début et à la fin URL
df["url"] = df["url"].str.strip()
# Conversion des labels texte en valeurs numériques
df["label"] = df["label"].map({"benign": 0, "malicious": 1}).astype(int) #Encodage des labels
print(df["label"].value_counts()) #Vérification de la distribution des classes Afichages
#La fonction .map() remplace les valeurs texte par des nombres selon le dictionnaire donné : "benign" → 0,"malicious" → 1
#.astype(int),On force le type de la colonne en entier (int)
#type d'encodage est utilisé:c est la forme de:Label Encoding On transforme des catégories texte en valeurs numériques
# Ce n’est pas fait avec LabelEncoder de sklearn Ici on utilise un mapping manuel avec un dictionnaire.
#Donc on peut dire :  C’est du Label Encoding manuel

label
0    345738
1    159195
Name: count, dtype: int64


Creation des 3 distributions dataset1(50% 50%),dataset2(90% 10%)et dataset(10%-90%)

In [ ]:
#Génère les trois distributions expérimentales à partir du même dataset source.
import pandas as pd
# Sépare le dataset en deux sous-ensembles selon la classe (bénigne / malveillante).
benign = df[df['label'] == 0]  #benign = DataFrame filtré contenant seulement les URLs bénignes
malicious = df[df['label'] == 1]
#Affiche le nombre d’échantillons par classe pour vérifier la disponibilité des données.
print(f"URLs bénignes disponibles: {len(benign)}")
print(f"URLs malveillantes disponibles: {len(malicious)}")  #len() calcule le nombre d’éléments dans la variable benign.
print(f"Total URLs disponibles: {len(df)}")

# Taille fixe pour tous les datasets\ to avoid 'Cannot take a larger sample than population error
TOTAL_SIZE = 176883

# ===== Dataset 1: Équilibré (50-50) =====
def create_balanced_dataset(benign, malicious, total_size=176883):
    """
    Crée un dataset équilibré avec 50% benign et 50% malicious
    """
    size_per_class = total_size // 2  #  176883 // 2 = 88441 pour chaque classe

    ben_sample = benign.sample(n=size_per_class, random_state=42)
    mal_sample = malicious.sample(n=size_per_class, random_state=42)

    dataset = pd.concat([ben_sample, mal_sample]) # Concatène  les deux DataFrames
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    return dataset

# ===== Dataset 2: 90% bénignes, 10% malveillantes =====
def create_imbalanced_dataset_90_10(benign, malicious, total_size=176883):
    """
    Crée un dataset avec 90% benign et 10% malicious
    """
    ben_size = int(total_size * 0.9)
    mal_size = int(total_size * 0.1)

    ben_sample = benign.sample(n=ben_size, random_state=42)
    mal_sample = malicious.sample(n=mal_size, random_state=42)

    dataset = pd.concat([ben_sample, mal_sample])
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    return dataset

# ===== Dataset 3: 10% bénignes, 90% malveillantes =====
def create_imbalanced_dataset_10_90(benign, malicious, total_size=176883):
    """
    Crée un dataset avec 10% benign et 90% malicious
    """
    ben_size = int(total_size * 0.1)
    mal_size = int(total_size * 0.9)

    ben_sample = benign.sample(n=ben_size, random_state=42)
    mal_sample = malicious.sample(n=mal_size, random_state=42)

    dataset = pd.concat([ben_sample, mal_sample])
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    return dataset

# ===== Créer les 3 datasets =====
print("\n" + "="*60)
print("CRÉATION DES DATASETS")
print("="*60)

dataset_balanced = create_balanced_dataset(benign, malicious, TOTAL_SIZE)
dataset_imb_90_10 = create_imbalanced_dataset_90_10(benign, malicious, TOTAL_SIZE)
dataset_imb_10_90 = create_imbalanced_dataset_10_90(benign, malicious, TOTAL_SIZE)

# ===== Afficher les statistiques =====
print("\n📊 DATASET 1 - BALANCED (50-50):")
print(f"   Total samples: {len(dataset_balanced)}")
print(f"   Distribution:")
print(dataset_balanced['label'].value_counts().sort_index())
print(f"   Pourcentages:")
print(dataset_balanced['label'].value_counts(normalize=True).sort_index() * 100)

print("\n📊 DATASET 2 - IMBALANCED (90% benign - 10% malicious):")
print(f"   Total samples: {len(dataset_imb_90_10)}")
print(f"   Distribution:")
print(dataset_imb_90_10['label'].value_counts().sort_index())
print(f"   Pourcentages:")
print(dataset_imb_90_10['label'].value_counts(normalize=True).sort_index() * 100)

print("\n📊 DATASET 3 - IMBALANCED (10% benign - 90% malicious):")
print(f"   Total samples: {len(dataset_imb_10_90)}")
print(f"   Distribution:")
print(dataset_imb_10_90['label'].value_counts().sort_index())
print(f"   Pourcentages:")
print(dataset_imb_10_90['label'].value_counts(normalize=True).sort_index() * 100)


URLs bénignes disponibles: 345738
URLs malveillantes disponibles: 159195
Total URLs disponibles: 504933

CRÉATION DES DATASETS

📊 DATASET 1 - BALANCED (50-50):
   Total samples: 176882
   Distribution:
label
0    88441
1    88441
Name: count, dtype: int64
   Pourcentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

📊 DATASET 2 - IMBALANCED (90% benign - 10% malicious):
   Total samples: 176882
   Distribution:
label
0    159194
1     17688
Name: count, dtype: int64
   Pourcentages:
label
0    90.000113
1     9.999887
Name: proportion, dtype: float64

📊 DATASET 3 - IMBALANCED (10% benign - 90% malicious):
   Total samples: 176882
   Distribution:
label
0     17688
1    159194
Name: count, dtype: int64
   Pourcentages:
label
0     9.999887
1    90.000113
Name: proportion, dtype: float64


#  Feature Engineering and Tokenization Pipeline (RoBERTa)

 Feature Engineering and Data Transformation - Loading RoBERTa Tokenizer, Implementing Tokenization Functions, Converting Between Dataset Formats, Tokenizing All Class Distribution Variants, and Preparing PyTorch-Compatible Tensors for Model Training

Charge le tokenizer RoBERTa pré-entraîné

In [ ]:
!pip install datasets #command permet d installer des bibliothèques.datasets: C’est la bibliothèque Hugging Face Datasets. Charger des datasets facilement+Manipuler de grandes données
!pip install transformers peft#Installe deux bibliothèques : transformers Bibliothèque Hugging Face pour:utiliser des modèles pré-entraînés (BERT, RoBERTa)+tokenizer.
#la bib PEFT = Parameter Efficient Fine-Tuning Permet de : Fine-tuner un modèle
import torch #Importe PyTorch sert à :gérer les tenseurs+entraîner les modèles deep learning+utiliser le GPU
from datasets import Dataset #Importe la classe Dataset de Hugging Face. Elle permet de : convertir un DataFrame pandas en dataset compatible Transformers
from transformers import RobertaTokenizerFast#Importe le tokenizer rapide de RoBERTa.

In [ ]:
#Charge le tokenizer RoBERTa pré-entraîné, qui convertit les URLs en tokens numériques exploitables par le modèle.
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")  #Ce bloc charge le tokenizer RoBERTa, mais pas le modèle lui-même.
#Confirme le chargement correct du tokenizer et affiche la taille de son vocabulaire.
print("Tokenizer chargé avec succès!")
print(f"Taille du vocabulaire: {tokenizer.vocab_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer chargé avec succès!
Taille du vocabulaire: 50265


Définir la fonction de tokenisation

In [ ]:
# ÉTAPE 2 : Définir la fonction de tokenisation pour tokeniser un batch de données
def tokenize_batch(batch):
    return tokenizer(
        batch["url"],#le texte (URLs) à convertir en tokens
        truncation=True,  #Si le texte est trop long, on ne garde que les 128 premiers tokens
        padding="max_length",  #padding="max_length" Tous les URL doivent avoir la même longueur+remplit les URLs courts jusqu’à max_length
        max_length=128 #limite les séquences à 128 tokens
    )
    #Cette fonction sert à transformer un groupe d’URLs en tokens numériques compréhensibles par le modèle RoBERTa.

 convertir les 3 dataFrames pandas en HuggingFace Dataset, RoBERTa + Trainer ne travaillent pas directement avec pandas.

In [ ]:
#Conversion des datasets Pandas en huggingface dataset
hf_balanced = Dataset.from_pandas(dataset_balanced)
hf_90_10 = Dataset.from_pandas(dataset_imb_90_10)  #(90% benign - 10% malicious)
hf_10_90 = Dataset.from_pandas(dataset_imb_10_90)  #(10% benign - 90% malicious)
#Pandas est utilisé pour nettoyer, analyser et préparer les données.
#Pandas est une bibliothèque utilisée principalement pour la manipulation et l’analyse des données.
#Hugging Face Dataset,est conçu pour le Deep Learning, en particulier pour NLP.
#Il permet de : Traiter les données en batch, Appliquer facilement la tokenisation avec .map()
#Être compatible avec les modèles Transformers+s’intégrer directement avec PyTorch et le Trainer
#Hugging Face Dataset est utilisé pour entraîner des modèles de Deep Learning,
# car il permet le traitement en batch et l’intégration directe avec les Transformers.


Le Hugging Face Dataset est converti en tenseurs PyTorch (avec set_format("torch")). Ensuite, lors de l'entraînement, les batches de données sont envoyés vers le GPU par PyTorch pour entraîner le modèle.

Appliquer le tokenizer RoBERTa aux 3 datasets, Obtenir des datasets prêts pour l’entraînement

In [ ]:
# Tokenisation du dataset équilibré (50–50)
hf_balanced = hf_balanced.map(
    tokenize_batch,#Applique la fonction tokenize_batch en batch sur le dataset Hugging Face.
    batched=True,#Traiter les données par lots (ex: 1000 lignes à la fois) au lieu d'une par une Taille du batch par défaut : 1000 lignes/ traitement vectorisé, plus rapide pour les grands datasets.
    remove_columns=["url"]# le modèle n’a besoin que des input_ids et attention_mask. on garde seulement les features nécessaires
)
#.map() applique la fonction tokenize_batch à chaque exemple (ou batch d’exemples) du dataset et retourne un nouveau dataset transformé.
#Taille du batch non précisée → Hugging Face utilise 1000 par défaut, mais peut être ajustée si GPU limité.

Map:   0%|          | 0/176882 [00:00<?, ? examples/s]

In [ ]:
#Tokenisation du dataset 10–90
hf_10_90 = hf_10_90.map(
    tokenize_batch,
    batched=True,
    remove_columns=["url"]
)

Map:   0%|          | 0/176882 [00:00<?, ? examples/s]

In [ ]:
#Tokenisation du dataset 90–10
hf_90_10 = hf_90_10.map(
    tokenize_batch,
    batched=True,
    remove_columns=["url"]
)

Map:   0%|          | 0/176882 [00:00<?, ? examples/s]

Définir le format PyTorch:Convertit les datasets HF en tenseurs PyTorch

In [ ]:
columns = ["input_ids", "attention_mask", "label"] #Spécifie les colonnes que le modèle RoBERTa utilisera pour l’entraînement
#Convertit les datasets HF en tenseurs PyTorch, prêts pour l’entraînement,  Ne garde que les colonnes spécifiées (input_ids, attention_mask, label) → optimisation mémoire.
hf_balanced.set_format(type="torch", columns=columns)
hf_90_10.set_format(type="torch", columns=columns)
hf_10_90.set_format(type="torch", columns=columns)

Affichage pour faire la vérification

In [ ]:
print(hf_balanced)
print(hf_90_10)
print(hf_10_90)

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 176882
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 176882
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 176882
})


# Stratified 5-Fold Cross-Validation, LoRA Fine-Tuning, and Model Evaluation

Séparation des données stratified k fold cross validation pour les 3 dataset\ k= 5

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Créer l'objet K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Extraire les labels pour la stratification
labels_balanced = np.array(hf_balanced['label'])
#hf_balanced['label']:retourne tous les labels sous forme de liste Python :StratifiedKFold attend un tableau de labels on converti en array
#np.array() sert à créer un tableau NumPy.
labels_90_10 = np.array(hf_90_10['label'])
labels_10_90 = np.array(hf_10_90['label'])
# StratifiedKFold : C’est une méthode de validation croisée (cross-validation) qui :Divise les données en kfolds+Conserve la même proportion des classes dans chaque fold
#StratifiedKFold garantit que chaque fold garde la même proportion de classes
#shuffle=True garantit que la répartition ne dépend pas de l’ordre initial des données

FINE-TUNING AVEC LoRA

Si tu entraînes le modèle sur balanced puis tu ré-entraîne sur 90/10 avec le même objet model, tu “contamines” l’expérience (carry-over).

➡️ Tu dois recréer un modèle neuf (mêmes poids init) pour chaque dataset.

In [ ]:
from peft import LoraConfig,TaskType,get_peft_model #outils pour appliquer LoRA
from transformers import RobertaForSequenceClassification
#Tokenizer:convertit le texte en tokens (input_ids, attention_mask) deja charger, On charge le modèle RoBERTa pour la classification binaire il reçoit ces tokens et produit des prédictions/ classification
def build_lora_roberta():
# Charger modèle pré-entraîné
    base = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2) #Définit num_labels=2 pour la classification binaire (benign/malicious).
#la taille du modèle est grande (~125M paramètres) → GPU nécessaire pour fine-tuning complet, LoRA réduit ce coût.
#Paramètres standard pour fine-tuning efficace avec LoRA (Configuration LoRA )
    lora_config = LoraConfig(
        r=8, #rang de la décomposition low-rank → contrôle la capacité d’adaptation
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "value"],#appliqué uniquement aux matrices Q et V des transformers
        bias="none",
        task_type=TaskType.SEQ_CLS # Sequence Classification, tâche de classification de séquences
    )
    #  Appliquer LoRA au modèle,Transforme le modèle RoBERTa en modèle LoRA, où seules certaines couches sont entraînables.
    model = get_peft_model(base, lora_config)

    return model

Préparer l’entraînement

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer

In [ ]:
# TrainingArguments : classe qui permet de définir tous les paramètres d’entraînement du modèle, Nombre d’epochs, taille des batches, learning rate, Stratégie d’évaluation et de sauvegarde GPU/FP16, logging, etc.
#Trainer : classe qui gère automatiquement l’entraînement, l’évaluation et la sauvegarde du modèle, automatisera tout l’entraînement : forward/backward pass, calcul de loss, optimisation, scheduler, checkpointing.
#Trainer appelle automatiquement  fonction compte metric à chaque évaluation (eval_dataset).
#output_dir doit être différent pour chaque expérience,Sinon tu écrases les résultats du balanced avec ceux du 90/10 etc.
training_args = TrainingArguments(
    output_dir="./results/balanced", # dossier pour sauvegarder les checkpoints
    num_train_epochs=1, #nombre de passages sur tout le dataset
    per_device_train_batch_size=16, # batch size pour l'entraînement
    per_device_eval_batch_size=32,# batch size pour évaluation
    learning_rate=5e-5,
    save_strategy="epoch", # sauvegarder modèle à chaque epoch
    eval_strategy="epoch", # Added evaluation strategy , évaluer à chaque fin d'epoch
    fp16=True  # activer float16 si GPU compatible, pour Optimisation GPU
)

Fonction d’évaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

In [ ]:
#On définit une fonction appelée automatiquement par le Trainer à chaque évaluation.
#Elle doit :recevoir les prédictions du modèle+retourner un dictionnaire de métriques
#calculer des métriques personnalisées (precision, recall, F1-score) pendant l’évaluation.
def compute_metrics(eval_pred):
    """
    Cette fonction est appelée par Trainer à chaque évaluation.
    eval_pred: tuple (logits, labels)
    Retourne: dictionnaire {"precision":..., "recall":..., "f1":...}
    """
    #Entrée : un tuple (logits, labels), Sortie : dictionnaire avec les métriques, compatible avec HF Trainer.
    logits, labels = eval_pred

    #eval_pred est un tuple contenant :logits+labels réels
#Les logits sont les sorties brutes du modèle, avant softmax
#exemple:[[ 2.3, -1.2], [ 0.1,  1.7], ...]

    # Convertir logits en prédictions de classe
    preds = torch.argmax(torch.tensor(logits), axis=1).numpy()
    labels = labels.astype(int)

    # Calcul des métriques
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

   #Cette fonction :
#Reçoit les sorties du modèle
#Convertit les logits en classes prédictes
#Compare avec les vraies étiquettes
#Calcule precision / recall / f1
#Retourne un dictionnaire compatible avec HF Trainer

In [ ]:
import torch
print(torch.cuda.is_available())  #entraînement sur GPU
#Cette fonction vérifie si CUDA est disponible sur ta machine. CUDA est la technologie de NVIDIA qui permet d’utiliser le GPU pour les calculs.


True


In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Aucun GPU CUDA détecté")



#|# Matériel | Temps attendu (1 epoch) |
#| --------  | ----------------------- |
#| CPU      |  20–100 h                |
#| GPU T4   | ~30–60 min               |
#| GPU V100 | ~20–40 min               |
#| GPU A100 | ~5–10 min                |



Tesla T4


#Trainer Setup, Training (3 Datasets), and Metrics Computation

Créer Trainer+ Lancer l'entraînement balanced dataset+calcul les metric

In [ ]:
from transformers import RobertaForSequenceClassification
# POUR DATASET BALANCED (50-50)
print("=" * 50)
print("ENTRAÎNEMENT AVEC 5-FOLD CV - DATASET BALANCED")
print("=" * 50)

fold_results_balanced = []#Liste vide qui va contenir les résultats de trainer.evaluate() pour chaque fold.
#Boucle sur les folds (5 itérations)
#stratifié = garde la même proportion de classes dans train/test du fold.
#enumerate() est une fonction Python intégrée qui permet de parcourir
# une séquence en récupérant à la fois :l’index (compteur)+la valeur de la case
#enumerate fold commance a 1 jusqu 5 fold
for fold, (train_idx, test_idx) in enumerate(skf.split(hf_balanced, labels_balanced), 1):
    print(f"\n--- FOLD {fold}/5 ---")

    # Sélectionner les données pour ce fold(Construire les datasets de ce fold train+test)
    train_fold = hf_balanced.select(train_idx)
    test_fold = hf_balanced.select(test_idx)

    # Créer un NOUVEAU modèle pour chaque fold
    model = build_lora_roberta() # on instancie un modèle neuf à chaque fold.

    # Créer Trainer pour ce fold
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )

    # Entraîner
    trainer.train()  #Lance l’entraînement sur train_fold+Utilise ce que tu as mis dans TrainingArguments :epochs, batch size, lr, fp16, save/eval strategy, etc.

    # Évaluer
    #est une méthode de la classe Trainer (Hugging Face Transformers).Elle sert à : Évaluer le modèle sur le dataset de test.
    eval_results = trainer.evaluate()#Évalue le modèle sur test_fold.
    fold_results_balanced.append(eval_results)#Ajoute les résultats de ce fold dans la liste.Après la boucle, tu pourras faire la moyenne des folds.

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}")

In [ ]:
# Calculer la moyenne des résultats balaned dataset
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_balanced])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_balanced])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_balanced])

print("\n" + "=" * 50)
print("RÉSULTATS MOYENS - BALANCED")
print("=" * 50)
print(f"F1 moyen: {avg_f1:.4f}")
print(f"Precision moyenne: {avg_precision:.4f}")
print(f"Recall moyen: {avg_recall:.4f}")

Lancer l'entraînement 90-10 dataset+ calcul les metric

In [ ]:
# POUR DATASET 90-10
print("\n\n" + "=" * 50)
print("ENTRAÎNEMENT AVEC 5-FOLD CV - DATASET 90-10")
print("=" * 50)

fold_results_90_10 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_90_10, labels_90_10), 1):
    print(f"\n--- FOLD {fold}/5 ---")

    train_fold = hf_90_10.select(train_idx)
    test_fold = hf_90_10.select(test_idx)

    model = build_lora_roberta()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_results = trainer.evaluate()
    fold_results_90_10.append(eval_results)

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}")

In [ ]:
# Calculer la moyenne des résultats pour dataset 90 10
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_90_10])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_90_10])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_90_10])

print("\n" + "=" * 50)
print("RÉSULTATS MOYENS - dataset (90-10)")
print("=" * 50)
print(f"F1 moyen (90-10): {avg_f1:.4f}")
print(f"Precision moyenne (90-10): {avg_precision:.4f}")
print(f"Recall moyen (90-10): {avg_recall:.4f}")

In [ ]:
# ============================================================================
# CALCUL PR-AUC - TOUS LES DATASETS BASELINE (sans affichage courbes)
# ============================================================================

from sklearn.metrics import precision_recall_curve, auc
import torch
import numpy as np

print("\n" + "="*70)
print("       CALCUL PR-AUC POUR TOUS LES DATASETS BASELINE")
print("="*70)

In [ ]:
# ============================================================================
# 1. PR-AUC pour BALANCED (50-50) BASELINE
# ============================================================================
print("\n[1/2] Calcul PR-AUC pour BALANCED (50-50) BASELINE...")

pr_auc_values_balanced_baseline = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_balanced, labels_balanced), 1):
    # Recréer le même split
    train_fold = hf_balanced.select(train_idx)
    test_fold = hf_balanced.select(test_idx)

    # Créer et entraîner le modèle baseline
    model = build_lora_roberta()
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )
    trainer.train()

    # Obtenir les probabilités
    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()  # Probabilités classe positive (malicious)
    y_true = predictions.label_ids

    # Calculer PR-AUC
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_balanced_baseline.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

# Moyenne PR-AUC
balanced_baseline_pr_auc = np.mean(pr_auc_values_balanced_baseline)
print(f"\n  📊 PR-AUC MOYEN BALANCED BASELINE: {balanced_baseline_pr_auc:.4f}")

In [ ]:
# ============================================================================
# 2. PR-AUC pour 90-10 BASELINE
# ============================================================================
print("\n[2/2] Calcul PR-AUC pour 90-10 BASELINE...")

pr_auc_values_90_10_baseline = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_90_10, labels_90_10), 1):
    # Recréer le même split
    train_fold = hf_90_10.select(train_idx)
    test_fold = hf_90_10.select(test_idx)

    # Créer et entraîner le modèle baseline
    model = build_lora_roberta()
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )
    trainer.train()

    # Obtenir les probabilités
    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    # Calculer PR-AUC
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_90_10_baseline.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

In [ ]:

# Moyenne PR-AUC
dataset_90_10_baseline_pr_auc = np.mean(pr_auc_values_90_10_baseline)
print(f"\n  📊 PR-AUC MOYEN 90-10 BASELINE: {dataset_90_10_baseline_pr_auc:.4f}")

# ============================================================================
# RÉSUMÉ DES RÉSULTATS BASELINE
# ============================================================================
print("\n" + "="*70)
print("           📊 RÉSUMÉ PR-AUC - TOUS DATASETS BASELINE")
print("="*70)
print(f"  BALANCED (50-50) BASELINE:  {balanced_baseline_pr_auc:.4f}")
print(f"  10-90 BASELINE:              {dataset_10_90_baseline_pr_auc:.4f}")
print(f"  90-10 BASELINE:              {dataset_90_10_baseline_pr_auc:.4f}")
print("="*70)

# Sauvegarder les valeurs pour comparaison ultérieure
print("\n✅ Valeurs PR-AUC baseline sauvegardées.")

In [ ]:
# ============================================================================
# 3. PR-AUC pour 10-90 BASELINE
# ============================================================================
print("\n[3/3] Calcul PR-AUC pour 10-90 BASELINE...")

pr_auc_values_10_90_baseline = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_10_90, labels_10_90), 1):
    train_fold = hf_10_90.select(train_idx)
    test_fold = hf_10_90.select(test_idx)

    model = build_lora_roberta()
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )
    trainer.train()

    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_10_90_baseline.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

dataset_10_90_baseline_pr_auc = np.mean(pr_auc_values_10_90_baseline)
print(f"\n  📊 PR-AUC MOYEN 10-90 BASELINE: {dataset_10_90_baseline_pr_auc:.4f}")


[3/3] Calcul PR-AUC pour 10-90 BASELINE...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

KeyboardInterrupt



Lancer l'entraînement 10-90 dataset+ calcul les metric

In [ ]:
# POUR DATASET 10-90
from collections import Counter
print("\n\n" + "=" * 50)
print("ENTRAÎNEMENT AVEC 10-FOLD CV - DATASET 10-90")
print("=" * 50)

fold_results_10_90 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_10_90, labels_10_90), 1):
    print(f"\n--- FOLD {fold}/10 ---")

    train_fold = hf_10_90.select(train_idx)
    test_fold = hf_10_90.select(test_idx)


    model = build_lora_roberta()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_results = trainer.evaluate()
    fold_results_10_90.append(eval_results)

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}")

In [ ]:
# Calculer la moyenne des résultats pour dataset 10 90
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_10_90])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_10_90])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_10_90])

print("\n" + "=" * 50)
print("RÉSULTATS MOYENS - dataset (10-90)")
print("=" * 50)
print(f"F1 moyen (10-90): {avg_f1:.4f}")
print(f"Precision moyenne (10-90): {avg_precision:.4f}")
print(f"Recall moyen (10-90): {avg_recall:.4f}")

#5-Fold Stratified CV with Class Weighting (3 Datasets)

Ajouter un  code définit un Trainer personnalisé basé sur la classe Trainer de Hugging Face Transformers, dont le rôle principal est de gérer le déséquilibre des classes lors de l’entraînement d’un modèle de classification

In [ ]:
from transformers import Trainer #Trainer → classe de base Hugging Face
import torch.nn as nn

class WeightedTrainer(Trainer):#Tu crées une classe qui hérite de Trainer.
#Elle garde tout le comportement normal du Trainer MAIS tu peux modifier certaines méthodes
    """
    Trainer personnalisé qui utilise des class weights
    pour gérer le déséquilibre des classes
    """
#__init__ est le constructeur d’une classe.
#args permet de recevoir tous les arguments positionnels.
#**kwargs permet de recevoir les arguments nommés. exemples:
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights


     #Tu redéfinis la façon dont la loss est calculée+Le Trainer appelle automatiquement cette fonction pendant l’entraînement.

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Surcharge la fonction de loss pour utiliser des weights
        """
        labels = inputs.pop("labels")

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculer la loss avec class weights
        if self.class_weights is not None:
            # Déplacer les weights sur le même device que le modèle
            weights = self.class_weights.to(logits.device)

            # CrossEntropyLoss avec weights
            loss_fct = nn.CrossEntropyLoss(weight=weights)
            loss = loss_fct(logits, labels)
        else:
            # Loss standard si pas de weights
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# DATASET 1 : BALANCED (50-50)

DATASET 1 : BALANCED (50-50)

5-Fold Stratified Cross-Validation with Class Weighting balanced dataset

Ce code met en œuvre une validation croisée stratifiée à 5 plis combinée à une approche d’apprentissage sensible au coût (Cost-Sensitive Learning) afin d’entraîner et d’évaluer un modèle de classification sur un jeu de données équilibré.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
#Dataset Balnced:Donc ici le weighted learning n’aura pratiquement aucun effet.


# Créer le K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
labels_balanced = np.array(hf_balanced['label'])

print("=" * 60)
print("COST-SENSITIVE LEARNING - DATASET BALANCED (50-50)")
print("=" * 60)

fold_results_balanced = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_balanced, labels_balanced), 1):
    print(f"\n--- FOLD {fold}/5 ---")

    # Sélectionner les données
    train_fold = hf_balanced.select(train_idx)
    test_fold = hf_balanced.select(test_idx)

    # Calculer class weights pour ce fold
    train_labels = np.array(train_fold['label'])#On calcule les weights uniquement sur les données d'entraînement du fold.
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    # on convertir weight en tensor
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    print(f"Class weights: Benign={class_weights[0]:.4f}, Malicious={class_weights[1]:.4f}")

    # Créer modèle
    model = build_lora_roberta()

    # Créer Trainer avec weights
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )

    # Entraîner et évaluer
    trainer.train()
    eval_results = trainer.evaluate()
    fold_results_balanced.append(eval_results)

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}, Recall: {eval_results['eval_recall']:.4f}")
 #À chaque itération de la boucle :
#On crée un nouveau train_fold
#On extrait ses labels
#On calcule les class weights à partir de CES labels
#les weights sont recalculés 5 fois (une fois par fold).

COST-SENSITIVE LEARNING - DATASET BALANCED (50-50)

--- FOLD 1/5 ---
Class weights: Benign=1.0000, Malicious=1.0000


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Fold 4 - F1: 0.9924, Recall: 0.9865

--- FOLD 5/5 ---
Class weights: Benign=1.0000, Malicious=1.0000


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.046720,0.050551,0.998166,0.984679,0.991377


Fold 5 - F1: 0.9914, Recall: 0.9847


In [ ]:
# Résultats moyens balanced dataset
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_balanced])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_balanced])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_balanced])

print("\n" + "=" * 60)
print("RÉSULTATS MOYENS - BALANCED")
print("=" * 60)
print(f"F1 moyen: {avg_f1:.4f}")
print(f"Recall moyen: {avg_recall:.4f}")
print(f"Precision moyenne: {avg_precision:.4f}")


RÉSULTATS MOYENS - BALANCED
F1 moyen: 0.9917
Recall moyen: 0.9852
Precision moyenne: 0.9982


In [ ]:
# ============================================================================
# 1. PR-AUC pour BALANCED (50-50) COST-SENSITIVE
# ============================================================================
print("\n[1/2] Calcul PR-AUC pour BALANCED (50-50) COST-SENSITIVE...")

pr_auc_values_balanced_cost = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_balanced, labels_balanced), 1):
    # Recréer le même split
    train_fold = hf_balanced.select(train_idx)
    test_fold = hf_balanced.select(test_idx)

    # Calculer class weights
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Créer et entraîner avec cost-sensitive
    model = build_lora_roberta()
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
    trainer.train()

    # Obtenir les probabilités
    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    # Calculer PR-AUC
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_balanced_cost.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

# Moyenne PR-AUC
balanced_cost_sensitive_pr_auc = np.mean(pr_auc_values_balanced_cost)
print(f"\n  📊 PR-AUC MOYEN BALANCED COST-SENSITIVE: {balanced_cost_sensitive_pr_auc:.4f}")

# DATASET 2 : 90% Benign - 10% Malicious

DATASET 2 : 90% Benign - 10% Malicious

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
labels_90_10 = np.array(hf_90_10['label'])

print("\n\n" + "=" * 60)
print("COST-SENSITIVE LEARNING - DATASET 90-10")
print("=" * 60)

# ✨ NOUVEAU : Définir le facteur d'amplification
AMPLIFICATION_FACTOR = 100.0  # ← CHANGEMENT 1 : Ajuster ce facteur (2, 3, 5, 10...)

fold_results_90_10 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_90_10, labels_90_10), 1):
    print(f"\n--- FOLD {fold}/5 ---")

    train_fold = hf_90_10.select(train_idx)
    test_fold = hf_90_10.select(test_idx)

    # Calculer class weights
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # ✨  Afficher les weights ORIGINAUX
    print(f"Weights originaux  - Benign: {class_weights[0]:.4f}, Malicious: {class_weights[1]:.4f}")

    # ✨ Identifier la classe minoritaire et l'amplifier
    # Pour 90-10 : classe 1 (malicious) est minoritaire
    # Pour 10-90 : classe 0 (benign) serait minoritaire

    # Compter les échantillons par classe
    count_class_0 = np.sum(train_labels == 0)
    count_class_1 = np.sum(train_labels == 1)

    # Amplifier la classe minoritaire
    if count_class_0 < count_class_1:
        # Classe 0 est minoritaire
        class_weights[0] = class_weights[0] * AMPLIFICATION_FACTOR
        print(f"Amplification classe 0 (minoritaire) x{AMPLIFICATION_FACTOR}")
    else:
        # Classe 1 est minoritaire
        class_weights[1] = class_weights[1] * AMPLIFICATION_FACTOR
        print(f"Amplification classe 1 (minoritaire) x{AMPLIFICATION_FACTOR}")

    # ✨  Afficher les weights APRÈS amplification
    print(f"Weights amplifiés  - Benign: {class_weights[0]:.4f}, Malicious: {class_weights[1]:.4f}")
    print(f"Ratio (Malicious/Benign): {class_weights[1]/class_weights[0]:.2f}x")

    model = build_lora_roberta()

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
   # VÉRIFIER que le Trainer a bien les weights
    print(f"✓ Weights dans le Trainer: {trainer.class_weights}")
    trainer.train()
    eval_results = trainer.evaluate()
    fold_results_90_10.append(eval_results)

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}, Recall: {eval_results['eval_recall']:.4f}, Precision: {eval_results['eval_precision']:.4f}")



COST-SENSITIVE LEARNING - DATASET 90-10

--- FOLD 1/5 ---
Weights originaux  - Benign: 0.5556, Malicious: 5.0002
Amplification classe 1 (minoritaire) x100.0
Weights amplifiés  - Benign: 0.5556, Malicious: 500.0177
Ratio (Malicious/Benign): 900.04x


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ WeightedTrainer initialisé avec weights: tensor([  0.5556, 500.0177])
✓ Compute_loss utilise weights: tensor([  0.5556, 500.0177], device='cuda:0')


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.063671,0.084950,0.986448,0.987564,0.987006


Fold 1 - F1: 0.9870, Recall: 0.9876, Precision: 0.9864

--- FOLD 2/5 ---
Weights originaux  - Benign: 0.5556, Malicious: 5.0002
Amplification classe 1 (minoritaire) x100.0
Weights amplifiés  - Benign: 0.5556, Malicious: 500.0177
Ratio (Malicious/Benign): 900.04x


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ WeightedTrainer initialisé avec weights: tensor([  0.5556, 500.0177])
✓ Compute_loss utilise weights: tensor([  0.5556, 500.0177], device='cuda:0')


Epoch,Training Loss,Validation Loss


In [ ]:
# Résultats moyens
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_90_10])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_90_10])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_90_10])

print("\n" + "=" * 60)
print("RÉSULTATS MOYENS - 90-10")
print("=" * 60)
print(f"F1 moyen: {avg_f1:.4f}")
print(f"Recall moyen: {avg_recall:.4f}")
print(f"Precision moyenne: {avg_precision:.4f}")


RÉSULTATS MOYENS - 90-10
F1 moyen: 0.9847
Recall moyen: 0.9861
Precision moyenne: 0.9833


In [ ]:
# CALCUL PR-AUC - COST-SENSITIVE + COURBES PR pour  dataset 90-10
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

print("\n" + "="*70)
print("    CALCUL PR-AUC COST-SENSITIVE + COURBES PR ( Dataset 90-10 uniquement)")
print("="*70)


    CALCUL PR-AUC COST-SENSITIVE + COURBES PR ( Dataset 90-10 uniquement)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_curve

# ============================================================================
# 2. PR-AUC + COURBES PR pour 90-10 COST-SENSITIVE
# ============================================================================
print("\n Calcul PR-AUC + COURBES PR pour 90-10 COST-SENSITIVE...")

# IMPORTANT: Utiliser le même AMPLIFICATION_FACTOR que dans votre cellule 56
AMPLIFICATION_FACTOR = 100.0  # ← Ajuster si vous avez changé dans cellule 56

pr_auc_values_90_10_cost = []
pr_curves_90_10_cost = []  # Pour stocker les courbes PR

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_90_10, labels_90_10), 1):
    # Recréer le même split
    train_fold = hf_90_10.select(train_idx)
    test_fold = hf_90_10.select(test_idx)

    # Calculer class weights avec amplification
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Amplifier la classe minoritaire (même logique que cellule 56)
    count_class_0 = np.sum(train_labels == 0)
    count_class_1 = np.sum(train_labels == 1)
    if count_class_0 < count_class_1:
        class_weights[0] = class_weights[0] * AMPLIFICATION_FACTOR
    else:
        class_weights[1] = class_weights[1] * AMPLIFICATION_FACTOR

    # Créer et entraîner avec cost-sensitive
    model = build_lora_roberta()
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
    trainer.train()

    # Obtenir les probabilités
    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    # Calculer precision, recall, thresholds
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)

    # Stocker les résultats
    pr_auc_values_90_10_cost.append(pr_auc)
    pr_curves_90_10_cost.append({
        'precision': precision,
        'recall': recall,
        'pr_auc': pr_auc,
        'fold': fold
    })

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")


 Calcul PR-AUC + COURBES PR pour 90-10 COST-SENSITIVE...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'WeightedTrainer' is not defined

In [ ]:
# Moyenne PR-AUC
dataset_90_10_cost_sensitive_pr_auc = np.mean(pr_auc_values_90_10_cost)
print(f"\n  📊 PR-AUC MOYEN 90-10 COST-SENSITIVE: {dataset_90_10_cost_sensitive_pr_auc:.4f}")



  📊 PR-AUC MOYEN 90-10 COST-SENSITIVE: 0.9918


In [ ]:
from sklearn.metrics import precision_recall_curve


# ============================================================================
# RÉCUPÉRER LES COURBES PR pour 90-10 BASELINE (déjà calculées précédemment)
# ============================================================================
print("\n[Récupération] Courbes PR pour 90-10 BASELINE (déjà calculées)...")

# Recalculer les courbes PR pour 90-10 baseline avec stockage des courbes
pr_curves_90_10_baseline = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_90_10, labels_90_10), 1):
    train_fold = hf_90_10.select(train_idx)
    test_fold = hf_90_10.select(test_idx)

    model = build_lora_roberta()
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics
    )
    trainer.train()

    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)

    pr_curves_90_10_baseline.append({
        'precision': precision,
        'recall': recall,
        'pr_auc': pr_auc,
        'fold': fold
    })

print("  ✓ Courbes 90-10 BASELINE récupérées.")


[Récupération] Courbes PR pour 90-10 BASELINE (déjà calculées)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.017509,0.017867,0.986445,0.987281,0.986863


NameError: name 'auc' is not defined

In [ ]:
# ============================================================================
# VISUALISATION : COURBES PR - 90-10 BASELINE vs COST-SENSITIVE
# ============================================================================
print("\n📊 Génération du graphique de comparaison pour Dataset 90-10 avant et apres...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ============================================================================
# Graphique 1: Courbes PR superposées
# ============================================================================
ax1.set_title('Courbes Precision-Recall - Dataset 90-10 BASELINE vs COST-SENSITIVE',
              fontsize=14, fontweight='bold')

# Tracer BASELINE (bleu)
for curve in pr_curves_90_10_baseline:
    ax1.plot(curve['recall'], curve['precision'],
             alpha=0.25, linewidth=1.5, color='#3498db')
ax1.plot([], [], color='#3498db', linewidth=3,
         label=f'BASELINE (PR-AUC={dataset_90_10_baseline_pr_auc:.4f})')

# Tracer COST-SENSITIVE (rouge)
for curve in pr_curves_90_10_cost:
    ax1.plot(curve['recall'], curve['precision'],
             alpha=0.25, linewidth=1.5, color='#e74c3c')
ax1.plot([], [], color='#e74c3c', linewidth=3,
         label=f'COST-SENSITIVE (PR-AUC={dataset_90_10_cost_sensitive_pr_auc:.4f})')

ax1.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax1.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left', fontsize=11)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])

In [ ]:

# ============================================================================
# Graphique 2: Barres de comparaison PR-AUC
# ============================================================================
methods = ['BASELINE\n(sans cost-sensitive)', 'COST-SENSITIVE\n(avec class weights)']
aucs = [dataset_90_10_baseline_pr_auc, dataset_90_10_cost_sensitive_pr_auc]
colors = ['#3498db', '#e74c3c']

bars = ax2.bar(methods, aucs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('PR-AUC Score', fontsize=12, fontweight='bold')
ax2.set_title('Comparaison PR-AUC - Dataset 90-10\nImpact du Cost-Sensitive Learning',
              fontsize=14, fontweight='bold')
ax2.set_ylim([0, 1.0])
ax2.grid(True, axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres
for bar, auc_val in zip(bars, aucs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{auc_val:.4f}',
             ha='center', va='bottom', fontweight='bold', fontsize=13)

# Ajouter une flèche d'amélioration
if improvement_90_10 > 0:
    ax2.annotate('', xy=(1, dataset_90_10_cost_sensitive_pr_auc),
                 xytext=(1, dataset_90_10_baseline_pr_auc),
                 arrowprops=dict(arrowstyle='->', lw=2.5, color='green'))
    ax2.text(1.15, (dataset_90_10_baseline_pr_auc + dataset_90_10_cost_sensitive_pr_auc)/2,
             f'+{improvement_90_10/dataset_90_10_baseline_pr_auc*100:.1f}%',
             fontsize=12, fontweight='bold', color='green')
elif improvement_90_10 < 0:
    ax2.annotate('', xy=(1, dataset_90_10_baseline_pr_auc),
                 xytext=(1, dataset_90_10_cost_sensitive_pr_auc),
                 arrowprops=dict(arrowstyle='->', lw=2.5, color='red'))
    ax2.text(1.15, (dataset_90_10_baseline_pr_auc + dataset_90_10_cost_sensitive_pr_auc)/2,
             f'{improvement_90_10/dataset_90_10_baseline_pr_auc*100:.1f}%',
             fontsize=12, fontweight='bold', color='red')

plt.tight_layout()
plt.show()

print("\n✅ Graphique de comparaison affiché.")
print("="*70)

# DATASET 3 : 10% Benign - 90% Malicious

DATASET 3 : 10% Benign - 90% Malicious

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
labels_10_90 = np.array(hf_10_90['label'])
print("\n\n" + "=" * 60)
print("COST-SENSITIVE LEARNING - DATASET 10-90")
print("=" * 60)
# ✨ NOUVEAU : Définir le facteur d'amplification
AMPLIFICATION_FACTOR = 100.0  # ← CHANGEMENT 1 : Ajuster ce facteur (2, 3, 5, 10...)

fold_results_10_90 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_10_90, labels_10_90), 1):
    print(f"\n--- FOLD {fold}/5 ---")

    train_fold = hf_10_90.select(train_idx)
    test_fold = hf_10_90.select(test_idx)

    # Calculer class weights
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)
  # ✨  Afficher les weights ORIGINAUX
    print(f"Weights originaux  - Benign: {class_weights[0]:.4f}, Malicious: {class_weights[1]:.4f}")

    # ✨ Identifier la classe minoritaire et l'amplifier
    # Pour 90-10 : classe 1 (malicious) est minoritaire
    # Pour 10-90 : classe 0 (benign) serait minoritaire

    # Compter les échantillons par classe
    count_class_0 = np.sum(train_labels == 0)
    count_class_1 = np.sum(train_labels == 1)

    # Amplifier la classe minoritaire
    if count_class_0 < count_class_1:
        # Classe 0 est minoritaire
        class_weights[0] = class_weights[0] * AMPLIFICATION_FACTOR
        print(f"Amplification classe 0 (minoritaire) x{AMPLIFICATION_FACTOR}")
    else:
        # Classe 1 est minoritaire
        class_weights[1] = class_weights[1] * AMPLIFICATION_FACTOR
        print(f"Amplification classe 1 (minoritaire) x{AMPLIFICATION_FACTOR}")

    # ✨  Afficher les weights APRÈS amplification
    print(f"Weights amplifiés  - Benign: {class_weights[0]:.4f}, Malicious: {class_weights[1]:.4f}")
    print(f"Ratio (Malicious/Benign): {class_weights[1]/class_weights[0]:.2f}x")
    model = build_lora_roberta()

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )

    trainer.train()
    eval_results = trainer.evaluate()
    fold_results_10_90.append(eval_results)

    print(f"Fold {fold} - F1: {eval_results['eval_f1']:.4f}, Recall: {eval_results['eval_recall']:.4f}")



COST-SENSITIVE LEARNING - DATASET 10-90

--- FOLD 1/5 ---
Class weights: Benign=5.0002, Malicious=0.5556


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.071763,0.040183,0.999777,0.987060,0.993378


Fold 1 - F1: 0.9934, Recall: 0.9871

--- FOLD 2/5 ---
Class weights: Benign=5.0002, Malicious=0.5556


...

Fold 4 - F1: 0.9929, Recall: 0.9861

--- FOLD 5/5 ---
Class weights: Benign=5.0002, Malicious=0.5556


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.032634,0.036075,0.999841,0.988441,0.994109


Fold 5 - F1: 0.9941, Recall: 0.9884


In [ ]:
# Calculer la moyenne des résultats pour dataset 10 90
avg_f1 = np.mean([r['eval_f1'] for r in fold_results_10_90])
avg_precision = np.mean([r['eval_precision'] for r in fold_results_10_90])
avg_recall = np.mean([r['eval_recall'] for r in fold_results_10_90])

print("\n" + "=" * 50)
print("RÉSULTATS MOYENS - dataset (10-90)")
print("=" * 50)
print(f"F1 moyen (10-90): {avg_f1:.4f}")
print(f"Precision moyenne (10-90): {avg_precision:.4f}")
print(f"Recall moyen (10-90): {avg_recall:.4f}")

In [ ]:
# ============================================================================
# 2. PR-AUC pour 10-90 COST-SENSITIVE
# ============================================================================
print("\n[2/3] Calcul PR-AUC pour 10-90 COST-SENSITIVE...")

pr_auc_values_10_90_cost = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_10_90, labels_10_90), 1):
    train_fold = hf_10_90.select(train_idx)
    test_fold = hf_10_90.select(test_idx)

    # Calculer class weights avec amplification
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Amplifier la classe minoritaire
    count_class_0 = np.sum(train_labels == 0)
    count_class_1 = np.sum(train_labels == 1)
    if count_class_0 < count_class_1:
        class_weights[0] = class_weights[0] * AMPLIFICATION_FACTOR
    else:
        class_weights[1] = class_weights[1] * AMPLIFICATION_FACTOR

    model = build_lora_roberta()
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
    trainer.train()

    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_10_90_cost.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

dataset_10_90_cost_sensitive_pr_auc = np.mean(pr_auc_values_10_90_cost)
print(f"\n  📊 PR-AUC MOYEN 10-90 COST-SENSITIVE: {dataset_10_90_cost_sensitive_pr_auc:.4f}")

In [ ]:
# ============================================================================
# 2. PR-AUC pour 10-90 COST-SENSITIVE
# ============================================================================
print("\n[2/3] Calcul PR-AUC pour 10-90 COST-SENSITIVE...")

pr_auc_values_10_90_cost = []

for fold, (train_idx, test_idx) in enumerate(skf.split(hf_10_90, labels_10_90), 1):
    train_fold = hf_10_90.select(train_idx)
    test_fold = hf_10_90.select(test_idx)

    # Calculer class weights avec amplification
    train_labels = np.array(train_fold['label'])
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Amplifier la classe minoritaire
    count_class_0 = np.sum(train_labels == 0)
    count_class_1 = np.sum(train_labels == 1)
    if count_class_0 < count_class_1:
        class_weights[0] = class_weights[0] * AMPLIFICATION_FACTOR
    else:
        class_weights[1] = class_weights[1] * AMPLIFICATION_FACTOR

    model = build_lora_roberta()
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=test_fold,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )
    trainer.train()

    predictions = trainer.predict(test_fold)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)
    y_probs = probs[:, 1].numpy()
    y_true = predictions.label_ids

    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    pr_auc_values_10_90_cost.append(pr_auc)

    print(f"  ✓ Fold {fold}/5 - PR-AUC: {pr_auc:.4f}")

dataset_10_90_cost_sensitive_pr_auc = np.mean(pr_auc_values_10_90_cost)
print(f"\n  📊 PR-AUC MOYEN 10-90 COST-SENSITIVE: {dataset_10_90_cost_sensitive_pr_auc:.4f}")

In [ ]:
# ============================================================================
# RÉSUMÉ FINAL DE TOUS LES PR-AUC
# ============================================================================
print("\n" + "="*70)
print("           📊 RÉSUMÉ COMPLET - PR-AUC DE TOUS LES DATASETS")
print("="*70)
print("\n🔵 BASELINE (sans cost-sensitive):")
print(f"   • BALANCED (50-50):  {balanced_baseline_pr_auc:.4f}")
print(f"   • 10-90:              {dataset_10_90_baseline_pr_auc:.4f}")
print(f"   • 90-10:              {dataset_90_10_baseline_pr_auc:.4f}")

print("\n🔴 COST-SENSITIVE (avec class weights):")
print(f"   • BALANCED (50-50):  {balanced_cost_sensitive_pr_auc:.4f}")
print(f"   • 10-90:              {dataset_10_90_cost_sensitive_pr_auc:.4f}")
print(f"   • 90-10:              {dataset_90_10_cost_sensitive_pr_auc:.4f}")

print("\n💡 AMÉLIORATIONS (Cost-Sensitive vs Baseline):")
improvement_balanced = balanced_cost_sensitive_pr_auc - balanced_baseline_pr_auc
improvement_10_90 = dataset_10_90_cost_sensitive_pr_auc - dataset_10_90_baseline_pr_auc
improvement_90_10 = dataset_90_10_cost_sensitive_pr_auc - dataset_90_10_baseline_pr_auc
print(f"   • BALANCED: {improvement_balanced:+.4f} ({improvement_balanced/balanced_baseline_pr_auc*100:+.2f}%)")
print(f"   • 10-90:     {improvement_10_90:+.4f} ({improvement_10_90/dataset_10_90_baseline_pr_auc*100:+.2f}%)")
print(f"   • 90-10:     {improvement_90_10:+.4f} ({improvement_90_10/dataset_90_10_baseline_pr_auc*100:+.2f}%)")
print("="*70)